# Notebook 06 — Leakage–domain interaction analysis

**Private execution notebook.** This notebook is part of the current Li–Na–K computational-materials workflow.

## Purpose

This notebook quantifies how target-adjacent feature access and chemical-domain separation interact.

It combines:

- Notebook 04 fold-level benchmark results across P0–P4;
- Notebook 05 physics-constrained multi-task results;
- Notebook 05's persisted record-level outer-test manifest.

## Main questions

1. How much do intentionally leaky protocols improve apparent error?
2. How much does performance worsen outside the random-split setting?
3. Does leakage mask the severity of framework, family, chemical-system, or working-ion shift?
4. Does the physics-constrained model reduce physical inconsistency, and at what predictive-error cost?

## Statistical policy

- Notebook 04 fold rows are the inferential units.
- Protocol comparisons are paired within target, model, split, and fold where possible.
- Domain-shift interaction intervals use bootstrap resampling of fold rows within each protocol–split cell.
- Chemical-system R² is not used as the primary interaction metric because many test groups are small and low-variance.
- MAE is the primary metric; RMSE, R², and Spearman are secondary.
- No DFT, candidate ranking, or manuscript writing is performed.

In [ ]:
from __future__ import annotations

import json
import hashlib
import os
import platform
import re
import shutil
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_SEED = 20260712
BOOTSTRAP_REPS = 2000

def _locate_repository_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for root in [candidate, *candidate.parents]:
        if (
            (root / "notebooks").is_dir()
            and (root / "data").is_dir()
            and (root / "results").is_dir()
            and (root / "provenance").is_dir()
        ):
            return root
    raise FileNotFoundError("Could not locate the clean-room repository root.")


REPOSITORY_ROOT = _locate_repository_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from software.cmt_repository_paths import artifact_namespace, runtime_cache_root

ROOT = REPOSITORY_ROOT
OUTPUT_ROOT = artifact_namespace("06", REPOSITORY_ROOT)
AUDIT_DIR = OUTPUT_ROOT / "audit"
METRICS_DIR = OUTPUT_ROOT / "metrics"
PROCESSED_DIR = OUTPUT_ROOT / "processed"
METADATA_DIR = OUTPUT_ROOT / "metadata"
LOG_DIR = OUTPUT_ROOT / "logs"

for d in [AUDIT_DIR, METRICS_DIR, PROCESSED_DIR, METADATA_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def utc_now():
    return datetime.now(timezone.utc).isoformat()

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Output:", OUTPUT_ROOT)

In [ ]:
# -
# Canonical clean-room input resolution
# -
import sys

def _locate_repository_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for root in [candidate, *candidate.parents]:
        if (
            (root / "notebooks").is_dir()
            and (root / "data").is_dir()
            and (root / "results").is_dir()
            and (root / "provenance").is_dir()
        ):
            return root
    raise FileNotFoundError("Could not locate the clean-room repository root.")


REPOSITORY_ROOT = _locate_repository_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from software.cmt_repository_paths import artifact_namespace, runtime_cache_root

ROOT = REPOSITORY_ROOT
NB10 = artifact_namespace("04", REPOSITORY_ROOT)
NB10B = artifact_namespace("05", REPOSITORY_ROOT)

required = {
    "nb10_decision": NB10 / "metadata" / "04_final_decision.json",
    "nb10_fold": NB10 / "processed" / "04_benchmark_fold_results.csv",
    "nb10_agg": NB10 / "processed" / "04_benchmark_aggregate_results.csv",
    "nb10b_decision": NB10B / "metadata" / "05_final_decision.json",
    "nb10b_predictions": NB10B / "processed" / "05_physics_constrained_oof_predictions.csv",
    "nb10b_consistency": NB10B / "metrics" / "05_physics_consistency_metrics.csv",
    "nb10b_fold_metrics": NB10B / "metrics" / "05_independent_vs_constrained_fold_metrics.csv",
    "nb10b_manifest": NB10B / "processed" / "05_outer_test_record_manifest.csv",
    "nb10b_split_audit": NB10B / "audit" / "05_split_reuse_audit.csv",
}

missing = [str(p) for p in required.values() if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(missing))


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


input_hashes = pd.DataFrame([
    {"role": role, "path": str(path.resolve()), "sha256": sha256(path)}
    for role, path in required.items()
])
input_hashes.to_csv(METADATA_DIR / "06_input_file_hashes.csv", index=False)

print("Notebook 04 namespace:", NB10)
print("Notebook 05 namespace:", NB10B)


In [ ]:
# -
# Preflight decisions and data loading
# -
decision10 = json.loads(required["nb10_decision"].read_text(encoding="utf-8"))
decision10b = json.loads(required["nb10b_decision"].read_text(encoding="utf-8"))

accepted10 = decision10.get("final_decision") in {
    "FULL_GO_TO_NOTEBOOK_11",
    "FULL_GO_TO_NOTEBOOK_10C",
}
accepted10b = decision10b.get("final_decision") == "FULL_GO_TO_NOTEBOOK_10C"

fold10 = pd.read_csv(required["nb10_fold"], low_memory=False)
agg10 = pd.read_csv(required["nb10_agg"], low_memory=False)
pred10b = pd.read_csv(required["nb10b_predictions"], low_memory=False)
cons10b = pd.read_csv(required["nb10b_consistency"], low_memory=False)
fold10b = pd.read_csv(required["nb10b_fold_metrics"], low_memory=False)
manifest10b = pd.read_csv(required["nb10b_manifest"], low_memory=False)
split_audit10b = pd.read_csv(required["nb10b_split_audit"], low_memory=False)

preflight = pd.DataFrame([
    {"check": "notebook10_decision_accepted", "pass": bool(accepted10)},
    {"check": "notebook10b_decision_accepted", "pass": bool(accepted10b)},
    {"check": "notebook10_no_error_rows", "pass": bool((fold10["status"] == "OK").all())},
    {"check": "notebook10b_split_audit_pass", "pass": bool(split_audit10b["pass"].all())},
    {"check": "notebook10b_record_manifest_nonempty", "pass": bool(len(manifest10b) > 0)},
])
preflight.to_csv(AUDIT_DIR / "06_preflight_audit.csv", index=False)

if not preflight["pass"].all():
    display(preflight)
    raise RuntimeError("Notebook 06 preflight failed")

print("Notebook 04 fold rows:", len(fold10))
print("Notebook 05 prediction rows:", len(pred10b))
print("Notebook 05 test-membership rows:", len(manifest10b))

In [ ]:
# -
# Coverage audit
# -
expected_protocols = {"P0", "P1", "P2", "P3", "P4"}
expected_splits = {
    "random_split",
    "framework_groupkfold",
    "leave_family_out",
    "leave_chemical_system_out",
    "leave_working_ion_out",
}
expected_models = {"DummyMean", "Ridge", "ExtraTrees"}

coverage_rows = []
for label, observed, expected in [
    ("protocols", set(fold10["protocol"].astype(str)), expected_protocols),
    ("splits", set(fold10["split_name"].astype(str)), expected_splits),
    ("models", set(fold10["model_name"].astype(str)), expected_models),
]:
    coverage_rows.append({
        "dimension": label,
        "observed": "|".join(sorted(observed)),
        "missing": "|".join(sorted(expected - observed)),
        "pass": expected.issubset(observed),
    })

coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(AUDIT_DIR / "06_coverage_audit.csv", index=False)
if not coverage["pass"].all():
    display(coverage)
    raise RuntimeError("Required Notebook 04 coverage is incomplete")

print(coverage)

In [ ]:
# -
# Paired protocol effects within the same target/model/split/fold
# -
ml = fold10.loc[
    (fold10["status"] == "OK")
    & (fold10["model_name"].isin(["Ridge", "ExtraTrees"]))
].copy()

metrics = ["mae", "rmse", "r2", "spearman"]
keys = ["target", "model_name", "split_name", "fold_id", "heldout_group"]

wide = ml.pivot_table(
    index=keys,
    columns="protocol",
    values=metrics,
    aggfunc="first",
)
wide.columns = [f"{metric}_{protocol}" for metric, protocol in wide.columns]
wide = wide.reset_index()

comparisons = []
for leaky in ["P4", "P0"]:
    for clean in ["P1", "P2", "P3"]:
        required_cols = [f"{m}_{leaky}" for m in metrics] + [f"{m}_{clean}" for m in metrics]
        subset = wide.dropna(subset=required_cols).copy()
        for row in subset.itertuples(index=False):
            rec = {k: getattr(row, k) for k in keys}
            rec.update({"leaky_protocol": leaky, "clean_protocol": clean})
            for m in metrics:
                leaky_val = getattr(row, f"{m}_{leaky}")
                clean_val = getattr(row, f"{m}_{clean}")
                # Positive delta for MAE/RMSE means the clean setting is harder.
                # Positive delta for R²/Spearman means the leaky setting looks better.
                if m in {"mae", "rmse"}:
                    rec[f"{m}_clean_minus_leaky"] = clean_val - leaky_val
                else:
                    rec[f"{m}_leaky_minus_clean"] = leaky_val - clean_val
            comparisons.append(rec)

paired_protocol = pd.DataFrame(comparisons)
paired_protocol.to_csv(
    PROCESSED_DIR / "06_paired_protocol_fold_effects.csv",
    index=False,
)

summary_rows = []
for group, g in paired_protocol.groupby(
    ["target", "model_name", "split_name", "leaky_protocol", "clean_protocol"],
    dropna=False,
):
    target, model, split, leaky, clean = group
    for col in [
        "mae_clean_minus_leaky",
        "rmse_clean_minus_leaky",
        "r2_leaky_minus_clean",
        "spearman_leaky_minus_clean",
    ]:
        x = pd.to_numeric(g[col], errors="coerce").dropna().to_numpy()
        if len(x) == 0:
            continue
        summary_rows.append({
            "target": target,
            "model_name": model,
            "split_name": split,
            "leaky_protocol": leaky,
            "clean_protocol": clean,
            "effect_metric": col,
            "n_paired_folds": len(x),
            "mean_effect": float(np.mean(x)),
            "median_effect": float(np.median(x)),
            "sd_effect": float(np.std(x, ddof=1)) if len(x) > 1 else np.nan,
            "positive_effect_fraction": float(np.mean(x > 0)),
        })

protocol_effects = pd.DataFrame(summary_rows)
protocol_effects.to_csv(
    METRICS_DIR / "06_leakage_main_effects.csv",
    index=False,
)

print("Paired protocol rows:", len(paired_protocol))

In [ ]:
# -
# Domain-shift main effects and leakage × domain interaction
# -
hard_splits = [
    "framework_groupkfold",
    "leave_family_out",
    "leave_chemical_system_out",
    "leave_working_ion_out",
]

cell = (
    ml.groupby(["target", "model_name", "protocol", "split_name"], as_index=False)
      .agg(
          mae_mean=("mae", "mean"),
          mae_median=("mae", "median"),
          rmse_mean=("rmse", "mean"),
          r2_mean=("r2", "mean"),
          spearman_mean=("spearman", "mean"),
          n_folds=("fold_id", "nunique"),
          n_test_median=("n_test", "median"),
      )
)

domain_rows = []
interaction_rows = []

for target in sorted(cell["target"].unique()):
    for model in ["Ridge", "ExtraTrees"]:
        sub = cell[(cell.target == target) & (cell.model_name == model)]
        for protocol in ["P0", "P1", "P2", "P3", "P4"]:
            random_row = sub[(sub.protocol == protocol) & (sub.split_name == "random_split")]
            if len(random_row) != 1:
                continue
            random_mae = float(random_row.iloc[0]["mae_mean"])
            for split in hard_splits:
                hard_row = sub[(sub.protocol == protocol) & (sub.split_name == split)]
                if len(hard_row) != 1:
                    continue
                hard_mae = float(hard_row.iloc[0]["mae_mean"])
                domain_rows.append({
                    "target": target,
                    "model_name": model,
                    "protocol": protocol,
                    "hard_split": split,
                    "random_mae": random_mae,
                    "hard_split_mae": hard_mae,
                    "mae_delta_hard_minus_random": hard_mae - random_mae,
                    "mae_ratio_hard_over_random": hard_mae / random_mae if random_mae else np.nan,
                })

        for leaky in ["P4", "P0"]:
            for clean in ["P1", "P2", "P3"]:
                for split in hard_splits:
                    def get_mae(protocol, split_name):
                        r = sub[(sub.protocol == protocol) & (sub.split_name == split_name)]
                        return float(r.iloc[0]["mae_mean"]) if len(r) == 1 else np.nan

                    l_rand = get_mae(leaky, "random_split")
                    l_hard = get_mae(leaky, split)
                    c_rand = get_mae(clean, "random_split")
                    c_hard = get_mae(clean, split)
                    vals = [l_rand, l_hard, c_rand, c_hard]
                    if not np.isfinite(vals).all():
                        continue

                    interaction = (c_hard - c_rand) - (l_hard - l_rand)
                    interaction_rows.append({
                        "target": target,
                        "model_name": model,
                        "leaky_protocol": leaky,
                        "clean_protocol": clean,
                        "hard_split": split,
                        "leaky_domain_penalty_mae": l_hard - l_rand,
                        "clean_domain_penalty_mae": c_hard - c_rand,
                        "leakage_domain_interaction_mae": interaction,
                        "interpretation": (
                            "leakage_masks_domain_shift"
                            if interaction > 0
                            else "no_masking_or_reverse"
                        ),
                    })

domain_effects = pd.DataFrame(domain_rows)
interactions = pd.DataFrame(interaction_rows)

domain_effects.to_csv(
    METRICS_DIR / "06_domain_shift_main_effects.csv",
    index=False,
)
interactions.to_csv(
    METRICS_DIR / "06_leakage_domain_interaction_effects.csv",
    index=False,
)

print("Domain-effect rows:", len(domain_effects))
print("Interaction rows:", len(interactions))

In [ ]:
# -
# Group-aware bootstrap intervals for interaction effects
# -
rng = np.random.default_rng(RANDOM_SEED)

def bootstrap_mean(values, reps=BOOTSTRAP_REPS):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    draws = np.empty(reps, dtype=float)
    for i in range(reps):
        draws[i] = np.mean(rng.choice(x, size=len(x), replace=True))
    return float(np.mean(x)), float(np.quantile(draws, 0.025)), float(np.quantile(draws, 0.975))

boot_rows = []
for row in interactions.itertuples(index=False):
    target = row.target
    model = row.model_name
    leaky = row.leaky_protocol
    clean = row.clean_protocol
    hard = row.hard_split

    def fold_values(protocol, split):
        return ml.loc[
            (ml.target == target)
            & (ml.model_name == model)
            & (ml.protocol == protocol)
            & (ml.split_name == split),
            "mae",
        ].dropna().to_numpy()

    l_rand = fold_values(leaky, "random_split")
    l_hard = fold_values(leaky, hard)
    c_rand = fold_values(clean, "random_split")
    c_hard = fold_values(clean, hard)

    if min(map(len, [l_rand, l_hard, c_rand, c_hard])) == 0:
        continue

    draws = np.empty(BOOTSTRAP_REPS, dtype=float)
    for i in range(BOOTSTRAP_REPS):
        lr = np.mean(rng.choice(l_rand, size=len(l_rand), replace=True))
        lh = np.mean(rng.choice(l_hard, size=len(l_hard), replace=True))
        cr = np.mean(rng.choice(c_rand, size=len(c_rand), replace=True))
        ch = np.mean(rng.choice(c_hard, size=len(c_hard), replace=True))
        draws[i] = (ch - cr) - (lh - lr)

    boot_rows.append({
        "target": target,
        "model_name": model,
        "leaky_protocol": leaky,
        "clean_protocol": clean,
        "hard_split": hard,
        "interaction_mae_point": float(row.leakage_domain_interaction_mae),
        "bootstrap_reps": BOOTSTRAP_REPS,
        "ci95_low": float(np.quantile(draws, 0.025)),
        "ci95_high": float(np.quantile(draws, 0.975)),
        "probability_interaction_positive": float(np.mean(draws > 0)),
        "ci_excludes_zero": bool(
            np.quantile(draws, 0.025) > 0 or np.quantile(draws, 0.975) < 0
        ),
    })

interaction_boot = pd.DataFrame(boot_rows)
interaction_boot.to_csv(
    METRICS_DIR / "06_group_bootstrap_intervals.csv",
    index=False,
)

print("Bootstrap interaction rows:", len(interaction_boot))

In [ ]:
# -
# Physics-constrained value analysis from Notebook 05
# -
direct = cons10b[
    cons10b["model_variant"].isin([
        "unconstrained_direct",
        "softconstrained_direct",
    ])
].copy()

phys_wide = direct.pivot_table(
    index=["protocol", "split_name", "fold_id", "heldout_group"],
    columns="model_variant",
    values=[
        "energy_consistency_mae",
        "stability_consistency_mae",
        "any_negative_prediction_rate",
    ],
    aggfunc="first",
)
phys_wide.columns = [f"{m}_{v}" for m, v in phys_wide.columns]
phys_wide = phys_wide.reset_index()

phys_wide["energy_consistency_reduction"] = (
    phys_wide["energy_consistency_mae_unconstrained_direct"]
    - phys_wide["energy_consistency_mae_softconstrained_direct"]
)
phys_wide["stability_consistency_reduction"] = (
    phys_wide["stability_consistency_mae_unconstrained_direct"]
    - phys_wide["stability_consistency_mae_softconstrained_direct"]
)
phys_wide["negative_rate_reduction"] = (
    phys_wide["any_negative_prediction_rate_unconstrained_direct"]
    - phys_wide["any_negative_prediction_rate_softconstrained_direct"]
)

phys_wide.to_csv(
    PROCESSED_DIR / "06_paired_physics_constraint_fold_effects.csv",
    index=False,
)

phys_summary = (
    phys_wide.groupby(["protocol", "split_name"], as_index=False)
    .agg(
        n_folds=("fold_id", "nunique"),
        energy_consistency_reduction_mean=("energy_consistency_reduction", "mean"),
        energy_consistency_reduction_median=("energy_consistency_reduction", "median"),
        stability_consistency_reduction_mean=("stability_consistency_reduction", "mean"),
        stability_consistency_reduction_median=("stability_consistency_reduction", "median"),
        negative_rate_reduction_mean=("negative_rate_reduction", "mean"),
        energy_improved_fraction=("energy_consistency_reduction", lambda x: float(np.mean(np.asarray(x) > 0))),
        stability_improved_fraction=("stability_consistency_reduction", lambda x: float(np.mean(np.asarray(x) > 0))),
    )
)
phys_summary.to_csv(
    METRICS_DIR / "06_physics_constraint_value_summary.csv",
    index=False,
)

print("Physics paired folds:", len(phys_wide))

In [ ]:
# -
# Chemical-system caution table
# -
chem = fold10[
    (fold10["status"] == "OK")
    & (fold10["split_name"] == "leave_chemical_system_out")
    & (fold10["model_name"].isin(["Ridge", "ExtraTrees"]))
].copy()

chem_diag = (
    chem.groupby(["target", "protocol", "model_name"], as_index=False)
    .agg(
        n_groups=("fold_id", "nunique"),
        n_test_min=("n_test", "min"),
        n_test_median=("n_test", "median"),
        n_test_max=("n_test", "max"),
        mae_median=("mae", "median"),
        mae_iqr=("mae", lambda x: float(np.nanquantile(x, 0.75) - np.nanquantile(x, 0.25))),
        r2_median=("r2", "median"),
        r2_iqr=("r2", lambda x: float(np.nanquantile(x, 0.75) - np.nanquantile(x, 0.25))),
        negative_r2_fraction=("r2", lambda x: float(np.mean(pd.to_numeric(x, errors="coerce") < 0))),
    )
)
chem_diag["primary_reporting_policy"] = (
    "Use pooled/weighted error and group-level MAE distributions; do not use unweighted mean group R2 as the main result."
)
chem_diag.to_csv(
    AUDIT_DIR / "06_chemical_system_reporting_caution.csv",
    index=False,
)

In [ ]:
# -
# Final gates, decision, manifests
# -
physics_gate = bool(
    (phys_wide["energy_consistency_reduction"] > 0).mean() > 0.50
    and (phys_wide["stability_consistency_reduction"] > 0).mean() > 0.50
)

gate_rows = [
    ("input_decisions_accepted", bool(accepted10 and accepted10b)),
    ("notebook10_all_rows_ok", bool((fold10["status"] == "OK").all())),
    ("required_protocols_splits_models_present", bool(coverage["pass"].all())),
    ("notebook10b_split_audit_passed", bool(split_audit10b["pass"].all())),
    ("paired_protocol_effects_nonempty", bool(len(paired_protocol) > 0)),
    ("domain_effects_nonempty", bool(len(domain_effects) > 0)),
    ("interaction_effects_nonempty", bool(len(interactions) > 0)),
    ("bootstrap_intervals_complete", bool(len(interaction_boot) == len(interactions))),
    ("physics_consistency_improved_in_majority_of_folds", physics_gate),
    ("record_level_manifest_available", bool(len(manifest10b) > 0)),
]
gates = pd.DataFrame(gate_rows, columns=["gate", "pass"])
gates.to_csv(AUDIT_DIR / "06_go_no_go_gate_audit.csv", index=False)

decision = (
    "FULL_GO_TO_EXACT_STATE_DFT_PREPARATION"
    if gates["pass"].all()
    else "HOLD_06_VALIDATION_FAILURE"
)

decision_payload = {
    "final_decision": decision,
    "run_timestamp_utc": utc_now(),
    "n_notebook10_fold_rows": int(len(fold10)),
    "n_paired_protocol_rows": int(len(paired_protocol)),
    "n_interaction_rows": int(len(interactions)),
    "n_bootstrap_rows": int(len(interaction_boot)),
    "physics_value_gate": physics_gate,
    "next_step": (
        "Proceed to corrected exact-state DFT preparation while preserving Notebooks 11–13 as completed evidence."
        if decision.startswith("FULL_GO")
        else "Inspect failed gates before proceeding."
    ),
    "no_dft_performed": True,
    "no_candidate_ranking_performed": True,
    "no_manuscript_writing_performed": True,
}
(METADATA_DIR / "06_final_decision.json").write_text(
    json.dumps(decision_payload, indent=2),
    encoding="utf-8",
)

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "bootstrap_reps": BOOTSTRAP_REPS,
    "random_seed": RANDOM_SEED,
}
(METADATA_DIR / "06_software_environment.json").write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)

# Output manifest excludes itself.
rows = []
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file() and path.name != "06_output_file_manifest.csv":
        rows.append({
            "relative_path": str(path.relative_to(OUTPUT_ROOT)).replace("\\", "/"),
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        })
manifest = pd.DataFrame(rows)
manifest.to_csv(METADATA_DIR / "06_output_file_manifest.csv", index=False)

display(gates)
print("FINAL DECISION:", decision)

if decision != "FULL_GO_TO_EXACT_STATE_DFT_PREPARATION":
    raise RuntimeError(decision)